In [ ]:
# ================================================================
# IS PHASE9F-E — HUMAN VASCULAR AUTHOR-ANNOTATED REFERENCE (R3 STRICT)
# GSE256493 | Adult control temporal lobe vascular/perivascular
# Run in Colab High-RAM
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, subprocess, time
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/MASTER_DEGREE/IS_COLAB/phase9f_e_human_vascular")
INPUT_DIR = DRIVE_ROOT / "input"
OUTPUT_DIR = DRIVE_ROOT / "results" / "R3_STRICT_AUTHOR"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RDS_NAME = "GSE256493_Adult_control_brain_temporal_lobe_unsorted_endothelial_and_perivascular_cells_seurat_object.rds.gz"
RDS_DRIVE = INPUT_DIR / RDS_NAME

LOCAL_DIR = Path("/content/is_phase9f_e")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
RDS_LOCAL = LOCAL_DIR / RDS_NAME
R_SCRIPT = LOCAL_DIR / "run_phase9f_e_human.R"

EXPECTED_BYTES = 3493366138
NOTEBOOK_VERSION = "R3_STRICT_AUTHOR_ANNOTATION"
GEO_ACCESSION = "GSE256493"
PMID = "38987604"
DOI = "10.1038/s41586-024-07493-y"
AUTHOR_ANNOTATION_COLUMN = ""

def mem_total_gib():
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal:"):
                return int(line.split()[1]) / 1024 / 1024
    return 0

ram = mem_total_gib()
disk = shutil.disk_usage("/content").free / 1024**3
print(f"RAM_TOTAL_GiB={ram:.2f}")
print(f"DISK_FREE_GiB={disk:.2f}")

if ram < 11.5:
    raise RuntimeError(f"At least ~12 GiB RAM required. Detected {ram:.1f} GiB RAM.")
print("MEMORY_MODE=", "STANDARD_12GB_OPTIMIZED" if ram < 20 else "HIGH_RAM")
if disk < 20:
    raise RuntimeError("At least 20 GiB free local disk is required.")


# Optional local swap: useful on standard ~12.7 GiB Colab.
# Colab runtime permissions can vary; failure is non-fatal.
def try_enable_swap(size_gib=12):
    try:
        active = subprocess.run(
            ["bash","-lc","swapon --show --noheadings"],
            capture_output=True, text=True
        )
        if active.stdout.strip():
            print("SWAP_ALREADY_ACTIVE")
            return
        print(f"Attempting {size_gib} GiB local swap...")
        cmds = [
            f"rm -f /content/phase9f_swapfile",
            f"fallocate -l {size_gib}G /content/phase9f_swapfile",
            "chmod 600 /content/phase9f_swapfile",
            "mkswap /content/phase9f_swapfile >/dev/null",
            "swapon /content/phase9f_swapfile",
        ]
        r = subprocess.run(
            ["bash","-lc"," && ".join(cmds)],
            capture_output=True, text=True
        )
        if r.returncode == 0:
            print("SWAP_ENABLED")
            subprocess.run(["bash","-lc","free -h"])
        else:
            print("SWAP_NOT_AVAILABLE; continuing without swap")
            print((r.stderr or r.stdout)[-1000:])
    except Exception as e:
        print("SWAP_SETUP_SKIPPED", type(e).__name__, str(e))

try_enable_swap(12)

# Reduce allocator fragmentation in the R subprocess.
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["R_GC_MEM_GROW"] = "0"

if not RDS_DRIVE.exists():
    raise FileNotFoundError(RDS_DRIVE)

drive_size = RDS_DRIVE.stat().st_size
print("RDS_DRIVE=", RDS_DRIVE)
print("RDS_BYTES=", drive_size)

if drive_size != EXPECTED_BYTES:
    raise RuntimeError(f"Unexpected RDS size: {drive_size} != {EXPECTED_BYTES}")

if not (RDS_LOCAL.exists() and RDS_LOCAL.stat().st_size == drive_size):
    if RDS_LOCAL.exists():
        RDS_LOCAL.unlink()
    t0 = time.time()
    shutil.copy2(RDS_DRIVE, RDS_LOCAL)
    print("COPY_SECONDS=", round(time.time()-t0,2))

if RDS_LOCAL.stat().st_size != drive_size:
    raise RuntimeError("Local copy size mismatch")

import hashlib
EXPECTED_SHA256 = "337c3d82c070c7ea71ad21114af847e5f4c76ab199e70c2ebcdce6b03db104dd"

def sha256_file(path, chunk=16*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

local_sha256 = sha256_file(RDS_LOCAL)
print("LOCAL_RDS_SHA256=", local_sha256)
if local_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "RDS SHA256 mismatch; stop before readRDS. "
        f"{local_sha256} != {EXPECTED_SHA256}"
    )

def run(cmd, check=True):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(cmd, check=check)

if shutil.which("Rscript") is None:
    run(["apt-get","update","-qq"])
    run(["apt-get","install","-y","-qq","r-base","r-base-dev",
         "libcurl4-openssl-dev","libssl-dev","libxml2-dev",
         "libfontconfig1-dev","libharfbuzz-dev","libfribidi-dev",
         "libfreetype6-dev","libpng-dev","libtiff-dev","libjpeg-dev"])

subprocess.run(["apt-get","update","-qq"], check=False)
subprocess.run(["apt-get","install","-y","-qq",
                "r-cran-matrix","r-cran-data.table","r-cran-rcpp","r-cran-rcppeigen"], check=False)

install_script = r'''
repos <- c(CRAN="https://cloud.r-project.org")

matrix_ok <- requireNamespace("Matrix", quietly=TRUE) &&
             packageVersion("Matrix") >= "1.6.4"

if (!matrix_ok) {
  if (getRversion() < "4.4.0") {
    install.packages(
      "https://cran.r-project.org/src/contrib/Archive/Matrix/Matrix_1.6-5.tar.gz",
      repos=NULL,
      type="source",
      Ncpus=2
    )
  } else {
    install.packages("Matrix", repos=repos, Ncpus=2)
  }
}

if (!requireNamespace("SeuratObject", quietly=TRUE)) {
  install.packages("SeuratObject", repos=repos, Ncpus=2)
}

if (!requireNamespace("data.table", quietly=TRUE)) {
  install.packages("data.table", repos=repos, Ncpus=2)
}

if (!requireNamespace("SeuratObject", quietly=TRUE)) {
  stop("SeuratObject installation failed")
}
if (!requireNamespace("Matrix", quietly=TRUE)) {
  stop("Matrix installation failed")
}

cat("R_VERSION=", R.version.string, "\n")
cat("SEURATOBJECT=", as.character(packageVersion("SeuratObject")), "\n")
cat("MATRIX=", as.character(packageVersion("Matrix")), "\n")
'''
subprocess.run(["Rscript","-e",install_script], check=True)

r_code = r'''
suppressPackageStartupMessages({
  library(SeuratObject)
  library(Matrix)
})

args <- commandArgs(trailingOnly=TRUE)
RDS <- args[[1]]
OUT <- args[[2]]
REQUESTED_ANNOTATION <- if (length(args) >= 3) trimws(args[[3]]) else ""
dir.create(OUT, recursive=TRUE, showWarnings=FALSE)

targets <- c("FGF5","ALDH2","SH3PXD2A","COL4A2","COL4A1","CALHM2","NEURL1","C4orf22","INA")

cat("============================================================\n")
cat("READ RDS\n")
cat("============================================================\n")

t0 <- proc.time()
obj <- readRDS(gzfile(RDS, open="rb"))
gc(verbose=FALSE)
elapsed <- proc.time()-t0

cat("READ_RDS_SECONDS=", elapsed[["elapsed"]], "\n", sep="")
cat("CLASS=", paste(class(obj), collapse=";"), "\n", sep="")
cat("CELLS=", ncol(obj), "\n", sep="")
cat("FEATURES=", nrow(obj), "\n", sep="")

assays <- Assays(obj)
default_assay <- tryCatch(DefaultAssay(obj), error=function(e) "")

assay_rows <- list()
for (assay_name in assays) {
  A <- obj[[assay_name]]
  layers <- tryCatch(Layers(A), error=function(e) character())
  assay_rows[[length(assay_rows)+1]] <- data.frame(
    assay=assay_name,
    default_assay=assay_name==default_assay,
    n_features=nrow(A),
    n_cells=ncol(A),
    layers=paste(layers,collapse=";"),
    stringsAsFactors=FALSE
  )
}
assay_audit <- do.call(rbind, assay_rows)
write.table(assay_audit, file.path(OUT,"HUMAN_ASSAY_AUDIT.tsv"),
            sep="\t",quote=FALSE,row.names=FALSE)
print(assay_audit)

md <- obj[[]]
md$.__CELL__. <- rownames(md)

vascular_terms <- paste(c("endothelial","pericyte","smooth","vascular","fibro","mural",
                          "arter","venous","capillary","immune","microgl","macroph","vlmc"),
                        collapse="|")

meta_rows <- list()
for (col in colnames(md)) {
  x <- md[[col]]
  ux <- unique(as.character(x[!is.na(x)]))
  n_unique <- length(ux)
  name_low <- tolower(col)
  score <- 0
  if (grepl("cell.?type|celltype|annotation",name_low)) score <- score+100
  if (grepl("subtype|sub.?cluster|cell.?class",name_low)) score <- score+80
  if (grepl("cluster",name_low)) score <- score+35
  if (n_unique>=3 && n_unique<=100) score <- score+20
  examples <- paste(head(ux,30),collapse=";")
  if (grepl(vascular_terms,tolower(examples))) score <- score+35
  if (grepl("ncount|nfeature|percent|orig.ident|sample|donor|batch",name_low)) score <- score-100
  if (col=="seurat_clusters") score <- score-20
  meta_rows[[length(meta_rows)+1]] <- data.frame(
    column=col,n_unique=n_unique,score=score,examples=examples,
    stringsAsFactors=FALSE
  )
}
meta_audit <- do.call(rbind,meta_rows)
meta_audit <- meta_audit[order(-meta_audit$score,meta_audit$n_unique),,drop=FALSE]
write.table(meta_audit,file.path(OUT,"HUMAN_METADATA_AUDIT.tsv"),
            sep="\t",quote=FALSE,row.names=FALSE)
print(head(meta_audit,25))

annotation_col <- NA_character_
selection_method <- NA_character_
preferred <- c("celltype","cell_type","cell.type","CellType","cellType",
               "cell_type_final","celltype_final","annotation","cell_annotation",
               "celltype_annotation","sub.celltype","sub_celltype","subtype",
               "Cell_type","Cell_Type","celltypes","cell_types")

if (nzchar(REQUESTED_ANNOTATION)) {
  hit <- colnames(md)[tolower(colnames(md))==tolower(REQUESTED_ANNOTATION)]
  if (length(hit) != 1) {
    write.table(
      data.frame(status="EXPLICIT_AUTHOR_ANNOTATION_COLUMN_NOT_UNIQUE",
                 requested=REQUESTED_ANNOTATION,
                 matches=paste(hit,collapse=";"),
                 stringsAsFactors=FALSE),
      file.path(OUT,"HUMAN_ANNOTATION_SELECTION.tsv"),
      sep="	",quote=FALSE,row.names=FALSE
    )
    stop("Explicit author annotation column was not found uniquely")
  }
  annotation_col <- hit[[1]]
  selection_method <- "EXPLICIT_COLUMN_OVERRIDE"
} else {
  exact_hits <- character()
  for (p in preferred) {
    hit <- colnames(md)[tolower(colnames(md))==tolower(p)]
    exact_hits <- c(exact_hits, hit)
  }
  exact_hits <- unique(exact_hits)
  if (length(exact_hits) == 1) {
    annotation_col <- exact_hits[[1]]
    selection_method <- "EXACT_PREFERRED_NAME"
  } else if (length(exact_hits) > 1) {
    write.table(
      data.frame(status="AUTHOR_ANNOTATION_COLUMN_AMBIGUOUS",
                 candidate_columns=paste(exact_hits,collapse=";"),
                 stringsAsFactors=FALSE),
      file.path(OUT,"HUMAN_ANNOTATION_SELECTION.tsv"),
      sep="	",quote=FALSE,row.names=FALSE
    )
    stop("Multiple exact author-style annotation columns found; set AUTHOR_ANNOTATION_COLUMN explicitly")
  }
}

if (is.na(annotation_col)) {
  candidates <- meta_audit[
    meta_audit$score>=80 &
    meta_audit$n_unique>=3 &
    meta_audit$n_unique<=200,,drop=FALSE]
  write.table(
    data.frame(status="AUTHOR_ANNOTATION_COLUMN_UNRESOLVED",
               candidate_columns=paste(head(candidates$column,20),collapse=";"),
               stringsAsFactors=FALSE),
    file.path(OUT,"HUMAN_ANNOTATION_SELECTION.tsv"),
    sep="	",quote=FALSE,row.names=FALSE
  )
  stop("Author annotation column unresolved. Review HUMAN_METADATA_AUDIT.tsv and set AUTHOR_ANNOTATION_COLUMN explicitly.")
}

annotation <- as.character(md[[annotation_col]])
annotation[is.na(annotation)|annotation==""] <- "UNANNOTATED"

n_celltypes <- length(unique(annotation))
if (n_celltypes < 3 || n_celltypes > 200) {
  stop(paste0("Selected annotation column has implausible category count: ",n_celltypes))
}
nonempty_labels <- unique(annotation[annotation!="UNANNOTATED"])
numeric_like <- if(length(nonempty_labels)>0)
  mean(grepl("^[0-9]+([._-][0-9]+)?$",nonempty_labels)) else 1
if (numeric_like > 0.80) {
  stop("Selected annotation column is predominantly numeric cluster labels, not an author cell-type annotation")
}

selection <- data.frame(
  annotation_column=annotation_col,
  selection_method=selection_method,
  requested_column=REQUESTED_ANNOTATION,
  n_celltypes=n_celltypes,
  status="AUTHOR_ANNOTATION_SELECTED_FAIL_CLOSED",
  stringsAsFactors=FALSE
)
write.table(selection,file.path(OUT,"HUMAN_ANNOTATION_SELECTION.tsv"),
            sep="	",quote=FALSE,row.names=FALSE)

cat("ANNOTATION_COLUMN=",annotation_col,"
",sep="")
cat("ANNOTATION_SELECTION_METHOD=",selection_method,"
",sep="")

annotation_tab <- sort(table(annotation), decreasing=TRUE)
annotation_counts <- data.frame(
  celltype=names(annotation_tab),
  n_cells=as.integer(annotation_tab),
  stringsAsFactors=FALSE
)

write.table(
  annotation_counts,
  file.path(OUT,"HUMAN_AUTHOR_ANNOTATION_COUNTS.tsv"),
  sep="\t",quote=FALSE,row.names=FALSE
)

print(annotation_counts)

if ("RNA" %in% assays) {
  assay_name <- "RNA"
} else {
  assay_name <- default_assay
}
if (!assay_name %in% assays) assay_name <- assays[[1]]

A <- obj[[assay_name]]
all_layers <- tryCatch(Layers(A),error=function(e)character())
count_layers <- grep("^counts($|\\.)",all_layers,value=TRUE)
if (length(count_layers)==0) {
  count_layers <- grep("count",all_layers,value=TRUE,ignore.case=TRUE)
}
if (length(count_layers)==0) {
  stop("No raw count layer found; normalized data will not be substituted for pseudobulk.")
}

cat("ASSAY=",assay_name,"\n",sep="")
cat("COUNT_LAYERS=",paste(count_layers,collapse=";"),"\n",sep="")

broad_class <- function(x) {
  y <- tolower(x)
  if (grepl("pericy",y)) return("Pericyte")
  if (grepl("smooth|vsmc|(^|[^a-z])smc([^a-z]|$)|mural",y)) return("Smooth_muscle_mural")
  if (grepl("fibro|fibroblast|vlmc|perivascular[ _-]*fibro",y)) return("Perivascular_fibroblast")
  if (grepl("endothelial|capillary|arterial|venous|(^|[^a-z])ec([^a-z]|$)",y)) return("Endothelial")
  if (grepl("microgl|macroph|myeloid|immune|monocyte|lymph|t.?cell|b.?cell",y)) return("Immune")
  if (grepl("astro",y)) return("Astrocyte")
  return("Other")
}

# Feature metadata fallback protects against Ensembl-ID rownames.
feature_meta <- tryCatch(A[[]], error=function(e) NULL)

symbol_cols <- character()
if (!is.null(feature_meta) && nrow(feature_meta) > 0) {
  symbol_cols <- grep(
    "gene|symbol|feature.?name",
    colnames(feature_meta),
    value=TRUE,
    ignore.case=TRUE
  )
}

resolve_target_hits <- function(layer_genes, target) {

  direct <- which(toupper(layer_genes) == toupper(target))

  if (length(direct) > 0) {
    return(list(hit=direct, method="ROWNAME_SYMBOL"))
  }

  if (!is.null(feature_meta) && length(symbol_cols) > 0) {

    for (cc in symbol_cols) {

      vv <- toupper(as.character(feature_meta[[cc]]))
      ii <- which(vv == toupper(target))

      if (length(ii) == 0) next

      ids <- rownames(feature_meta)[ii]
      mm <- match(ids, layer_genes)
      mm <- mm[!is.na(mm)]

      if (length(mm) > 0) {
        return(list(
          hit=unique(mm),
          method=paste0("FEATURE_METADATA_",cc)
        ))
      }
    }
  }

  return(list(hit=integer(), method="UNRESOLVED"))
}


# Count layers should partition cells; overlap would double-count.
layer_cell_list <- lapply(
  count_layers,
  function(ll) {
    mm <- LayerData(A, layer=ll)
    cc <- colnames(mm)
    rm(mm)
    gc(verbose=FALSE)
    cc
  }
)

all_layer_cells <- unlist(layer_cell_list, use.names=FALSE)
duplicated_layer_cells <- unique(
  all_layer_cells[duplicated(all_layer_cells)]
)

layer_overlap_audit <- data.frame(
  n_count_layers=length(count_layers),
  total_layer_cell_entries=length(all_layer_cells),
  unique_cells=length(unique(all_layer_cells)),
  duplicated_cells_across_layers=length(duplicated_layer_cells),
  status=if(length(duplicated_layer_cells)==0)
    "PASS_DISJOINT_LAYERS"
  else
    "FAIL_OVERLAPPING_LAYERS",
  stringsAsFactors=FALSE
)

write.table(
  layer_overlap_audit,
  file.path(OUT,"HUMAN_COUNT_LAYER_OVERLAP_AUDIT.tsv"),
  sep="\t",quote=FALSE,row.names=FALSE
)

print(layer_overlap_audit)

if (length(duplicated_layer_cells) > 0) {
  stop("Count layers overlap in cell membership; aggregation would double-count cells.")
}


result_rows <- list()
gene_presence_rows <- list()

for (layer_name in count_layers) {
  cat("\nProcessing layer: ",layer_name,"\n",sep="")
  M <- LayerData(A,layer=layer_name)
  layer_cells <- colnames(M)
  if (is.null(layer_cells)) stop(paste("Layer has no cell names:",layer_name))
  idx_md <- match(layer_cells,rownames(md))
  if (any(is.na(idx_md))) stop(paste("Cell metadata mismatch in layer",layer_name))
  layer_annotation <- annotation[idx_md]
  layer_genes <- rownames(M)

  # Prefer author-provided nCount_RNA to avoid scanning the full sparse matrix.
  if ("nCount_RNA" %in% colnames(md)) {

    lib_size <- suppressWarnings(
      as.numeric(md[idx_md,"nCount_RNA"])
    )

    if (any(!is.finite(lib_size)) || any(lib_size < 0)) {
      lib_size <- Matrix::colSums(M)
      lib_size_source <- "MATRIX_COLSUMS"
    } else {
      lib_size_source <- "METADATA_nCount_RNA"
    }

  } else {

    lib_size <- Matrix::colSums(M)
    lib_size_source <- "MATRIX_COLSUMS"

  }

  celltypes <- sort(unique(layer_annotation))

  for (target in targets) {

    resolved <- resolve_target_hits(layer_genes,target)
    hit <- resolved$hit
    present <- length(hit)>=1

    gene_presence_rows[[length(gene_presence_rows)+1]] <- data.frame(
      target_gene=target,
      assay=assay_name,
      layer=layer_name,
      present=present,
      resolution_method=resolved$method,
      matched_features=if(present) paste(layer_genes[hit],collapse=";") else "",
      library_size_source=lib_size_source,
      stringsAsFactors=FALSE
    )
    if (!present) next

    target_vector <- Matrix::colSums(M[hit,,drop=FALSE])

    for (ct in celltypes) {
      cells_idx <- which(layer_annotation==ct)
      if (length(cells_idx)==0) next
      x <- target_vector[cells_idx]
      ls <- lib_size[cells_idx]
      target_sum <- sum(x)
      library_sum <- sum(ls)
      detected <- sum(x>0)

      result_rows[[length(result_rows)+1]] <- data.frame(
        gene=target,assay=assay_name,layer=layer_name,
        annotation_column=annotation_col,celltype=ct,
        broad_class=broad_class(ct),n_cells=length(cells_idx),
        target_sum_counts=target_sum,detected_cells=detected,
        detection_fraction=detected/length(cells_idx),
        library_size=library_sum,
        pseudobulk_cpm=if(library_sum>0) target_sum/library_sum*1e6 else 0,
        mean_raw_count=mean(x),stringsAsFactors=FALSE
      )
    }
  }

  rm(M)
  gc(verbose=FALSE)
}

presence <- do.call(rbind,gene_presence_rows)
write.table(presence,file.path(OUT,"HUMAN_TARGET_GENE_PRESENCE.tsv"),
            sep="\t",quote=FALSE,row.names=FALSE)

localization <- do.call(rbind,result_rows)
if (is.null(localization) || nrow(localization)==0) {
  stop("None of the target genes could be quantified.")
}

group_pairs <- unique(localization[,c("gene","celltype")])
combined <- list()

for (i in seq_len(nrow(group_pairs))) {
  gene <- group_pairs$gene[[i]]
  ct <- group_pairs$celltype[[i]]
  d <- localization[localization$gene==gene & localization$celltype==ct,,drop=FALSE]
  n_cells <- sum(d$n_cells)
  target_sum <- sum(d$target_sum_counts)
  detected <- sum(d$detected_cells)
  lib <- sum(d$library_size)

  combined[[length(combined)+1]] <- data.frame(
    gene=gene,celltype=ct,broad_class=broad_class(ct),
    n_cells=n_cells,target_sum_counts=target_sum,detected_cells=detected,
    detection_fraction=detected/n_cells,library_size=lib,
    pseudobulk_cpm=if(lib>0) target_sum/lib*1e6 else 0,
    stringsAsFactors=FALSE
  )
}

combined <- do.call(rbind,combined)
combined <- combined[order(combined$gene,-combined$detection_fraction,-combined$pseudobulk_cpm),,drop=FALSE]
write.table(combined,file.path(OUT,"HUMAN_TARGET_CELLTYPE_LOCALIZATION.tsv"),
            sep="\t",quote=FALSE,row.names=FALSE)

gene_summary <- list()
for (gene in targets) {
  d <- combined[combined$gene==gene,,drop=FALSE]
  if (nrow(d)==0) {
    gene_summary[[length(gene_summary)+1]] <- data.frame(
      gene=gene,present=FALSE,top_celltype="",top_broad_class="",
      top_n_cells=NA,
      top_detection_fraction=NA,top_pseudobulk_cpm=NA,
      ranking_scope="NOT_PRESENT",
      total_detected_cells=0,
      stringsAsFactors=FALSE
    )
    next
  }
  # Prevent tiny classes (e.g. 1/1 detected) from ranking first.
  rankable <- d[d$n_cells >= 50,,drop=FALSE]
  ranking_scope <- "N_CELLS_GE_50"

  if (nrow(rankable) == 0) {
    rankable <- d
    ranking_scope <- "ALL_CELLTYPES_FALLBACK"
  }

  rankable <- rankable[
    order(-rankable$detection_fraction,-rankable$pseudobulk_cpm),
    ,drop=FALSE
  ]

  top <- rankable[1,]

  gene_summary[[length(gene_summary)+1]] <- data.frame(
    gene=gene,
    present=TRUE,
    top_celltype=top$celltype,
    top_broad_class=top$broad_class,
    top_n_cells=top$n_cells,
    top_detection_fraction=top$detection_fraction,
    top_pseudobulk_cpm=top$pseudobulk_cpm,
    ranking_scope=ranking_scope,
    total_detected_cells=sum(d$detected_cells),
    stringsAsFactors=FALSE
  )
}
gene_summary <- do.call(rbind,gene_summary)
write.table(gene_summary,file.path(OUT,"HUMAN_TARGET_GENE_SUMMARY.tsv"),
            sep="\t",quote=FALSE,row.names=FALSE)

broad_rows <- list()
for (gene in unique(combined$gene)) {
  for (bc in unique(combined$broad_class)) {
    d <- combined[combined$gene==gene & combined$broad_class==bc,,drop=FALSE]
    if (nrow(d)==0) next
    cells <- sum(d$n_cells)
    detected <- sum(d$detected_cells)
    target_sum <- sum(d$target_sum_counts)
    lib <- sum(d$library_size)
    broad_rows[[length(broad_rows)+1]] <- data.frame(
      gene=gene,broad_class=bc,n_cells=cells,
      detection_fraction=detected/cells,
      pseudobulk_cpm=if(lib>0) target_sum/lib*1e6 else 0,
      stringsAsFactors=FALSE
    )
  }
}
broad <- do.call(rbind,broad_rows)
broad <- broad[order(broad$gene,-broad$detection_fraction,-broad$pseudobulk_cpm),,drop=FALSE]
write.table(broad,file.path(OUT,"HUMAN_TARGET_BROAD_CLASS.tsv"),
            sep="\t",quote=FALSE,row.names=FALSE)

branches <- data.frame(
  gene=c("FGF5","ALDH2","SH3PXD2A","COL4A2","COL4A1","CALHM2","NEURL1","C4orf22","INA"),
  genetic_branch=c(
    "REGULATORY_EXPRESSION_PROTEIN",
    "RS671_CODING_PROTEIN_METABOLIC",
    "CELL_SPECIFIC_REGULATORY",
    "VASCULAR_STRUCTURAL_REGULATORY",
    "SECONDARY_VASCULAR_STRUCTURAL",
    "SECONDARY_MOLECULAR_COLOC",
    "SECONDARY_MOLECULAR_COLOC",
    "SECONDARY_MOLECULAR_COLOC",
    "SECONDARY_MOLECULAR_COLOC"
  ),
  human_reference_role=c(
    "PRIMARY_LOCALIZATION",
    "SUPPORTIVE_LOCALIZATION_ONLY",
    "PRIMARY_CELLTYPE_LOCALIZATION",
    "PRIMARY_VASCULAR_LOCALIZATION",
    "COMPARE_WITH_COL4A2",
    "ORTHOGONAL_LOCALIZATION",
    "ORTHOGONAL_LOCALIZATION",
    "ORTHOGONAL_LOCALIZATION",
    "ORTHOGONAL_LOCALIZATION"
  ),
  stringsAsFactors=FALSE
)

branch_master <- merge(branches,gene_summary,by="gene",all.x=TRUE)
write.table(branch_master,file.path(OUT,"HUMAN_FUNCTIONAL_BRANCH_MASTER.tsv"),
            sep="\t",quote=FALSE,row.names=FALSE)

readiness <- data.frame(
  component=c("PHASE9F_E_HUMAN","RDS_PARSE","AUTHOR_ANNOTATION",
              "AUTHOR_ANNOTATION_METHOD","TARGET_GENE_LOCALIZATION",
              "HUMAN_REFERENCE_TYPE","INFERENCE_LEVEL","NEXT"),
  status=c("COMPLETE","PASS",paste0("PASS_COLUMN_",annotation_col),
           selection_method,"COMPLETE",
           "ADULT_CONTROL_TEMPORAL_LOBE_VASCULAR_PERIVASCULAR",
           "REFERENCE_LOCALIZATION_NOT_DISEASE_DGE",
           "INTEGRATE_HUMAN_WITH_MOUSE_AND_GENETIC_EVIDENCE"),
  stringsAsFactors=FALSE
)

write.table(readiness,file.path(OUT,"PHASE9F_E_HUMAN_READINESS.tsv"),
            sep="\t",quote=FALSE,row.names=FALSE)

report <- c(
  "# IS Phase9F-E Human Vascular Reference","",
  paste0("- Object class: ",paste(class(obj),collapse=", ")),
  paste0("- Cells: ",ncol(obj)),
  paste0("- Features: ",nrow(obj)),
  paste0("- Selected assay: ",assay_name),
  paste0("- Author annotation column: ",annotation_col),"",
  "## Target gene localization",""
)
for (i in seq_len(nrow(gene_summary))) {
  x <- gene_summary[i,]
  report <- c(report,paste0(
    "- ",x$gene,": present=",x$present,
    "; top_celltype=",x$top_celltype,
    "; broad_class=",x$top_broad_class,
    "; top_n_cells=",x$top_n_cells,
    "; detection=",signif(x$top_detection_fraction,4),
    "; CPM=",signif(x$top_pseudobulk_cpm,4)
  ))
}
writeLines(report,file.path(OUT,"PHASE9F_E_HUMAN_REPORT.md"))

print(gene_summary)
print(broad)
print(readiness)
cat("\nPHASE9F_E_HUMAN=PASS\n")
'''

R_SCRIPT.write_text(r_code)

cmd = ["Rscript", str(R_SCRIPT), str(RDS_LOCAL), str(OUTPUT_DIR), AUTHOR_ANNOTATION_COLUMN]
proc = subprocess.run(cmd, text=True, env=os.environ.copy())
if proc.returncode != 0:
    raise RuntimeError(f"R analysis failed: RC={proc.returncode}")

required = [
    "HUMAN_ASSAY_AUDIT.tsv",
    "HUMAN_METADATA_AUDIT.tsv",
    "HUMAN_ANNOTATION_SELECTION.tsv",
    "HUMAN_AUTHOR_ANNOTATION_COUNTS.tsv",
    "HUMAN_COUNT_LAYER_OVERLAP_AUDIT.tsv",
    "HUMAN_TARGET_GENE_PRESENCE.tsv",
    "HUMAN_TARGET_CELLTYPE_LOCALIZATION.tsv",
    "HUMAN_TARGET_GENE_SUMMARY.tsv",
    "HUMAN_TARGET_BROAD_CLASS.tsv",
    "HUMAN_FUNCTIONAL_BRANCH_MASTER.tsv",
    "PHASE9F_E_HUMAN_READINESS.tsv",
    "PHASE9F_E_HUMAN_REPORT.md",
]

missing=[]
for fn in required:
    p=OUTPUT_DIR/fn
    print("PASS" if p.exists() else "MISSING", fn, p.stat().st_size if p.exists() else "")
    if not p.exists():
        missing.append(fn)

if missing:
    raise RuntimeError("Missing outputs: "+", ".join(missing))

import json
import pandas as pd
from IPython.display import display

summary = pd.read_csv(OUTPUT_DIR/"HUMAN_TARGET_GENE_SUMMARY.tsv", sep="\t")
broad = pd.read_csv(OUTPUT_DIR/"HUMAN_TARGET_BROAD_CLASS.tsv", sep="\t")
readiness = pd.read_csv(OUTPUT_DIR/"PHASE9F_E_HUMAN_READINESS.tsv", sep="\t")
selection = pd.read_csv(OUTPUT_DIR/"HUMAN_ANNOTATION_SELECTION.tsv", sep="\t")

annotation_column = str(selection.loc[0,"annotation_column"])
selection_method = str(selection.loc[0,"selection_method"])

freeze_rows = [
    ("PHASE9F_E_HUMAN","COMPLETE"),
    ("DATASET",GEO_ACCESSION),
    ("PMID",PMID),
    ("DOI",DOI),
    ("INPUT_RDS",RDS_NAME),
    ("INPUT_BYTES",str(drive_size)),
    ("INPUT_SHA256",local_sha256),
    ("ANNOTATION_COLUMN",annotation_column),
    ("ANNOTATION_SELECTION_METHOD",selection_method),
    ("REFERENCE_TYPE","ADULT_CONTROL_TEMPORAL_LOBE_VASCULAR_PERIVASCULAR"),
    ("INFERENCE_LEVEL","REFERENCE_LOCALIZATION_NOT_DISEASE_DGE"),
    ("NEXT_GATE","MASTEROMICS_HUMAN_ANNOTATION_BINDING"),
]
freeze = pd.DataFrame(freeze_rows, columns=["component","status"])
freeze.to_csv(OUTPUT_DIR/"HUMAN_AUTHOR_ANNOTATION_FREEZE.tsv", sep="\t", index=False)

manifest = {
    "schema_version": 1,
    "phase": "PHASE9F_E_HUMAN",
    "notebook_version": NOTEBOOK_VERSION,
    "geo_accession": GEO_ACCESSION,
    "pmid": PMID,
    "doi": DOI,
    "input_rds": RDS_NAME,
    "input_bytes": int(drive_size),
    "input_sha256": local_sha256,
    "annotation_column": annotation_column,
    "annotation_selection_method": selection_method,
    "reference_type": "ADULT_CONTROL_TEMPORAL_LOBE_VASCULAR_PERIVASCULAR",
    "inference_level": "REFERENCE_LOCALIZATION_NOT_DISEASE_DGE",
    "target_genes": summary["gene"].astype(str).tolist(),
    "required_r_outputs": required,
    "freeze_file": "HUMAN_AUTHOR_ANNOTATION_FREEZE.tsv",
}
(OUTPUT_DIR/"HUMAN_RUN_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n"
)

for fn in ["HUMAN_AUTHOR_ANNOTATION_FREEZE.tsv","HUMAN_RUN_MANIFEST.json"]:
    p=OUTPUT_DIR/fn
    if not p.exists() or p.stat().st_size == 0:
        raise RuntimeError("Missing/empty freeze output: "+fn)
    print("PASS",fn,p.stat().st_size)

display(summary)
display(broad)
display(readiness)
display(freeze)

print("PHASE9F_E_HUMAN=COMPLETE")
print("NOTEBOOK_VERSION=", NOTEBOOK_VERSION)
print("OUTPUT_DIR=", OUTPUT_DIR)
